# 第五章：用一条多资产动量规则研究 Alpha

看到某个资产过去三个月上涨，最自然的猜测是它接下来还会涨；同样自然的另一个猜测是它已经涨得太多、接下来会跌。这两句话都听起来合理。研究的任务，是将其中一句变成明确规则，再用具有相同时间和成本口径的证据判断它。

本章只实现一个主线策略：**每天收盘后，观察股票、国债、黄金、商品四类教学资产过去 60 个交易日的涨跌；上涨持多头，下跌持空头，按过去波动分配风险，在下一开盘调整仓位。** 我们反复回到这条规则，解释机制、标准化、排序、信息系数、收益归因、成本和失效状态。其他策略在这条主线上说明“究竟替换了什么假设”，而不是另列一批缩写让读者记忆。

## 0. 学习目标与实验约定

读完后，你应能分清方向预测与相对排序、信号分数与组合权重、毛收益与风险调整后的增量；能从一个明确机制构造最小基线，诊断它在资产、期限和成本上的表现；能解释为什么同一个信号可以有正的信息系数却无法赚钱，也能有正收益却没有 Alpha。

全部代码只依赖 NumPy，使用固定种子的合成价格。教学资产允许无摩擦做空，稍后显式加入交易与借券成本；它们没有真实期货换月、股票分红、交易所涨跌停和融资安排。资产名称表达风险类别，不意味着这些合成收益是真实市场样本。主线的结果只能检验代码表达了什么，不能提供动量在现实中有效的证据。

## 1. 先说清收益可能从哪里来

趋势延续的一种机制是信息并非同时传给所有投资者：企业盈利或宏观变化被逐步确认，机构受流程、风险预算和交易成本约束，可能分批调仓。另一种机制是风险管理本身会放大方向：价格下跌、波动上升后，部分投资者被迫减仓。我们的 60 日规则试图捕捉这些渐进过程留下的价格延续。

机制必须同时说出失败方式。消息如果立即被价格充分吸收，过去涨跌就没有增量；方向反复切换时，趋势规则会不断追涨杀跌；市场突然反转时，过去 60 日的判断来不及调整；大量资金使用同类规则时，拥挤退出还会加剧冲击。“可能渐进调仓”是一条可检验的解释，不是正收益的保证。

Alpha（阿尔法）通常指相对某组基准风险暴露后剩余的平均收益。Beta（贝塔）是对这些共同风险的敏感度。风险溢价（risk premium）是承担某种系统性风险所获得的补偿；它可能长期为正，但也可能伴随少数非常痛苦的状态。噪声（noise）则包括有限样本中不可稳定复现的波动。这些不是互斥标签：同一条动量规则可能同时有市场暴露、风格溢价和噪声。

例如，主线在四个资产普遍上涨时恰好全部做多，赚到的钱可能主要来自承担共同风险。只有先与同资产集合的等权持有组合比较，再观察条件暴露和样本外表现，才有资格讨论增量。“策略赚钱”回答账户是否增长；“有 Alpha”还要回答增长相对什么、控制了哪些风险、扣了哪些成本。

## 2. 造一份足够小、时间含义完整的数据

每天有一个开盘价与一个收盘价。先模拟开盘到收盘的日间收益，再模拟收盘到下一开盘的隔夜收益，两者共同生成下一开盘。特征只读取已经结束的收盘记录；策略收益从下一开盘开始。这种构造让“看到的最后价格”和“真正能成交的价格”保持分离。

合成数据有几段不同的漂移方向，并叠加共同与独立扰动。漂移（drift）指收益条件均值的缓慢方向，扰动使单日结果围绕它波动。这样会人为制造一些趋势和反转，是为了让诊断有对象，不是为了估计真实市场收益。后续所有参数都固定，不能因为观察到某段亏损而悄悄改变数据生成机制。


In [ ]:
import numpy as np

rng = np.random.default_rng(20261027)
assets = np.array(["股票", "国债", "黄金", "商品"])
n_days, n_assets = 420, len(assets)
base_daily_vol = np.array([0.012, 0.005, 0.010, 0.015])
regimes = np.array([[0.0007, 0.0002, -0.0002, 0.0006],
                    [-0.0008, 0.0004, 0.0005, -0.0005],
                    [0.0006, -0.0003, 0.0004, 0.0008]])
open_price = np.empty((n_days + 1, n_assets))
close_price = np.empty((n_days, n_assets))
open_price[0] = [100, 100, 100, 100]
for t in range(n_days):
    drift = regimes[min(t // 140, 2)]
    common = rng.normal(0, 0.002)
    intraday = drift + common * np.array([1.0, -0.3, 0.2, 0.8])
    intraday += rng.normal(size=n_assets) * base_daily_vol * 0.75
    overnight = rng.normal(size=n_assets) * base_daily_vol * 0.40
    close_price[t] = open_price[t] * (1 + intraday)
    open_price[t + 1] = close_price[t] * (1 + overnight)
close_returns = close_price[1:] / close_price[:-1] - 1
print("价格形状:", open_price.shape, close_price.shape)
print("前五日股票开盘价:", np.round(open_price[:5, 0], 3))


## 3. 从过去涨跌到时间序列动量

时间序列动量（time-series momentum）判断每个资产相对它自身历史的方向。设 \(C_{i,t}\) 为资产 \(i\) 在第 \(t\) 日已经结束的收盘价，\(L=60\) 为回看长度，则

\[
m_{i,t}=C_{i,t}/C_{i,t-L}-1,\qquad d_{i,t}=\operatorname{sign}(m_{i,t}).
\]

\(m\) 是累计简单收益，\(d\) 是方向，正值对应多头、负值对应空头、恰为零时空仓。多头是先买后卖，价格上涨获利；空头是借入资产卖出，再买回归还，价格下跌获利。主线用有正负号的权重近似空头，实际做空是否有券可借、保证金是否足够、损失是否受限，都需要额外模型。

假如四个资产过去 60 日都跌了，本策略会四个都做空；假如都涨了，会四个都做多。它没有“必须有赢家和输家”的要求，因此天然不保证市场中性。图 5-1 先在单一真实指数上展示方向切换。指数价格本身不是可直接买卖的一股证券，图中多空状态也没有自动解决指数产品、借贷、融资和成本问题。




![真实 S&P 500 上 12 个月趋势规则的仓位状态与切换点](figures/ch05_trend_real_spx.png)

**图 5-1｜真实价格上的趋势规则。** 规则按 12 个月动量的符号决定多空状态，并滞后一个交易日执行；蓝色背景只标出多头区间，下图同时显示空头与暖启动空仓状态。负动量时做空，不是简单离场。图中波动与持仓时间不等于扣费后投资业绩。数据类型：公开 S&P 500 快照上的教学规则。


相同 10% 的累计涨幅，对低波动国债和高波动商品并不等价。我们估计最近 20 个收盘到收盘收益的日波动 \(\hat\sigma_{i,t}\)，再构造便于比较的分数

\[
s_{i,t}=\frac{m_{i,t}}{\hat\sigma_{i,t}\sqrt L}.
\]

分母是按平方根时间近似得到的 \(L\) 日波动，分数因而没有量纲。它不是严格的显著性统计量：收益可能自相关、波动会变化，累计简单收益也不完全等于日收益之和。这里用它表达“这段涨幅相对于该资产通常的波动有多大”，而不把 2 机械解释成某个置信水平。

主线权重仍保留最简单的方向规则：先令 \(u_{i,t}=d_{i,t}/\hat\sigma_{i,t}\)，再按 \(\sum_j|u_{j,t}|\) 归一化，最后逐项截到 40%。\(u\) 是临时风险缩放量，不是预测收益；\(w\) 才是最终资产名义金额占账户权益的比例。低波动资产获得更大名义仓位，是在试图让每个方向承担较接近的单资产波动风险。相关性尚未进入这里，所以它不是完整风险平价。

截断后不再强行补满：如果国债原本该占 55%，把它压到 40% 会让总绝对权重小于 1；保留未使用风险比再次归一化更容易保证上限。总绝对权重叫总敞口（gross exposure），有符号权重之和叫净敞口（net exposure）。四个各 25% 的多头总敞口和净敞口都是 1；两个 +25%、两个 -25% 的总敞口仍是 1，净敞口为零。


In [ ]:
def signal_at(t, lookback=60):
    momentum = close_price[t] / close_price[t - lookback] - 1
    # close_returns[k] 是 k 日收盘到 k+1 日收盘；此窗口只到 t 日。
    daily_vol = close_returns[t - 20:t].std(axis=0, ddof=1)
    daily_vol = np.maximum(daily_vol, 0.002)  # 教学风险下限，防止近零估计爆仓。
    score = momentum / (daily_vol * np.sqrt(lookback))
    return momentum, daily_vol, score

def weights_at(t, lookback=60, mode="trend"):
    momentum, daily_vol, score = signal_at(t, lookback)
    if mode == "trend":
        raw = np.sign(momentum) / daily_vol
    elif mode == "relative":
        ranks = np.array([np.sum(score < value) + (np.sum(score == value) - 1) / 2
                          for value in score])
        raw = (ranks - ranks.mean()) / daily_vol
    else:
        raise ValueError("未知信号映射")
    denominator = np.abs(raw).sum()
    weight = raw / denominator if denominator else np.zeros(n_assets)
    return np.clip(weight, -0.40, 0.40)

example_t = 180
momentum, daily_vol, score = signal_at(example_t)
example_w = weights_at(example_t)
for a, m, v, s, w in zip(assets, momentum, daily_vol, score, example_w):
    print(a, f"动量={m:.2%} 日波动={v:.2%} 分数={s:.2f} 权重={w:.2%}")
print("总敞口:", round(np.abs(example_w).sum(), 3), "净敞口:", round(example_w.sum(), 3))


图 5-2 将同类分数画成资产与时间的热图。主线也可以构造同样的矩阵：每个决策日一行、每个资产一列。连续同号的块意味着方向持续；所有资产同时翻转，则提示所谓分散可能只是共同风险在变。热图的颜色是信号强弱，不能直接读取成后续收益，也不能把高相关的几条信号算作几份独立证据。




![项目 A 合成面板上各资产各周的动量/波动信号热图](figures/ch05_signal_heatmap.png)

**图 5-2｜信号在时间和资产上的结构。** 热图每一行是一个资产、每一列是一周，颜色是动量除以波动后的信号值：红色看多、蓝色看空。可以看到信号在时间上成块持续（趋势状态），在资产间并不同步（分散化来源）。数据类型：projects/alpha_research 的确定性合成面板（seed 固定），不是真实市场历史。


## 4. 截面动量：替换问题以后，交易也变了

截面动量（cross-sectional momentum）问的是同一时刻谁相对更强。若四个资产过去分别涨 12%、8%、4%、1%，时间序列规则会全做多；截面规则可以做多涨 12% 的资产、做空只涨 1% 的资产。后一笔空头不是预测它一定会跌，而是预测它会跑输赢家。赢家涨 3%、输家涨 1% 时，多空价差仍能获利。

主线提供 `relative` 模式作为清楚的对照：按分数排名、减去平均排名、再做波动缩放。减去平均排名时原始分数和为零，但除以不同资产的波动后，权重和通常不再是零。若明确要求金额中性（dollar neutral），必须在最终权重上重新施加约束；金额中性也不等于贝塔中性（beta neutral），因为股票与国债的市场敏感度不同。

图 5-3 展示排序与权重的关系。排序只确定相对方向，权重还依赖波动、集中度和约束。改用截面动量后，研究对象已经变化，不能在看过曲线后把表现好的模式改叫原策略；两个模式应作为预先声明的不同假设报告。




![一个再平衡日的截面动量排序与截断归一化后的目标权重](figures/ch05_rebalance_weights.png)

**图 5-3｜截面排序如何变成权重。** 左图为一个再平衡日的 60 日动量排序。右图先用波动缩放信号，再按绝对信号和归一，截断单资产权重至 \(\pm25\%\)。截断后总绝对权重可能小于 1；单资产上限不能代替组合相关性管理。数据类型：项目 A 的确定性合成面板。


## 5. 如果方向反过来：均值回归究竟假设了什么

均值回归（mean reversion）不是简单把动量符号取反。它需要说明哪个经济量应该回到什么水平。两种可互换资产的价差可能受套利约束；同一股票短时过度反应后可能回补；而一家持续恶化公司的价格没有义务回到去年的高点。对主线股票“过去跌了所以买入”，隐含的稳定关系远弱于“两个可交割同一资产的价格偏离后可套利”。

配对交易（pairs trading）常写价差 \(x_t=P_{A,t}-\hat\beta P_{B,t}\)。这里 \(P_A,P_B\) 是两资产价格，\(\hat\beta\) 是训练期估计的对冲比率，使组合表达相对关系。若价差历史均值为 \(\mu\)、波动为 \(\sigma_x\)，则 \(z_t=(x_t-\mu)/\sigma_x\) 表示偏离程度。\(z=2\) 只说明相对过去均值高出两个历史标准差；若关系已经改变，它完全可以继续升到 5。

协整（cointegration）讨论非平稳价格的某个线性组合是否平稳，相关性则描述同向变化程度，二者不能替换。即使过去价差平稳，商业模式、融资条件或制度变化也可能使关系断裂。图 5-4 的后半段就是这种情况。主线若换成价差回归策略，就需要额外估计与验证对冲比率、稳定均值、持有期限和两边成交可行性，不能沿用动量策略的经济解释。




![合成价差的 z-score 与 ±2 触发带，后段出现结构断裂](figures/ch05_mean_reversion_zscore.png)

**图 5-4｜均值回归会失败的样子。** 前 450 天价差按均值回归过程生成，之后取消回归项并加入漂移。滚动 z-score 仍可能发出极值信号，但已不能依赖原来的平稳性假设；图中没有计算交易损益，不能据此断言某策略必然亏损。数据类型：固定 seed 合成价差。


## 6. 其他收益故事如何接入同一条研究链

价值（value）用价格与基本面比较，例如利润收益率 \(E/P\)：\(E\) 是每股利润，\(P\) 是股价，比值高表示每单位价格对应更多当期利润。便宜不等于一定会涨，可能是市场预期利润将下降。将主线动量换成价值时，行情输入要增加时点可见的财报，回看与再平衡周期往往变慢；上一章的版本与公告时间随之成为核心。

持有收益（carry）指在某些市场状态保持不变时，持有头寸随时间获得的收益组成。债券息票、外汇利差、期货曲线变化和卖出期权的时间价值，都可提供这类直觉，但机制不同。期货展期收益来自平旧合约、开新合约和曲线形状，不是简单看到新合约便宜就获得无风险现金；卖期权收取的权利金则伴随尾部赔付责任。若主线加一列 carry 特征，应同时加入融资、展期或期权暴露，不能把它当作免费的收益常数。

事件驱动（event-driven）围绕财报、并购或政策等明确事件交易。价格响应的是“实际相对事前预期的差异”，不只是数值本身。利润增长 20%，若市场原本预期 50%，也可能是坏消息。要研究惊喜（surprise），必须保留事件前可见的预期版本与准确发布时间；收盘形成的 60 日信号可以每日更新，公告策略则可能需要逐事件更新、处理跳空和停牌。

做市（market making）同时挂买卖报价，试图赚取价差。库存风险是成交后持有单边头寸所受的价格风险；逆向选择（adverse selection）是更知情的人在你的报价落后时主动与你成交。主线的日线数据无法观察报价队列和撤单，不能通过“每天赚半个价差”模拟做市。换成这个主题需要订单簿、事件级时间和成交概率模型。

这些策略可以互补，但名称不同不代表收益独立。价值与动量可能在某段时期都偏向同一行业，carry 与卖波动可能在危机中同时亏损。主线要增加第二个信号时，首先检查它是否在成本后改善组合，而不是把两个看似不同的英文名字直接相加。

## 7. 先量信号：信息系数到底在问什么

信息系数（information coefficient，IC）是同一决策时刻各资产分数与未来收益的相关性。秩信息系数（rank IC）先分别排名再计算相关性，减少极端值和分数尺度的影响。设 \(s_{i,t}\) 是收盘后得到的分数，\(y^{(h)}_{i,t}\) 是下一开盘到再过 \(h\) 个交易日开盘的收益，则

\[
IC_t^{(h)}=\operatorname{Corr}_{i}\bigl(\operatorname{rank}(s_{i,t}),\operatorname{rank}(y^{(h)}_{i,t})\bigr).
\]

下标 \(i\) 说明相关性在当日的资产截面上计算，\(t\) 是决策日，\(h\) 是持有期限。它与把单一股票数百天的数据拼起来计算时间序列相关性不同。只有四个资产时，每期排名非常粗，一次极端变化就能改变结果；本例适合学计算定义，不适合设定“IC 大于某个常数就可交易”的门槛。

手算例子：分数排名为 1、2、3、4，未来收益排名为 2、1、4、3，四个无并列名次差的平方和为 4。Spearman 相关为 \(1-6\times4/[4(4^2-1)]=0.6\)。这个较高的数既没有告诉我们涨跌的绝对幅度，也没有扣成本。赢家只是少跌一点时，截面排序很好，全部做多仍可能亏损。


In [ ]:
def average_ranks(values):
    values = np.asarray(values)
    order = np.argsort(values, kind="stable")
    ranks = np.empty(len(values), dtype=float)
    k = 0
    while k < len(values):
        end = k + 1
        while end < len(values) and values[order[end]] == values[order[k]]:
            end += 1
        ranks[order[k:end]] = (k + end - 1) / 2
        k = end
    return ranks

def rank_ic(scores, returns):
    x, y = average_ranks(scores), average_ranks(returns)
    if x.std() == 0 or y.std() == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

for h in [1, 5, 10, 20]:
    values = []
    for t in range(60, n_days - h):
        _, _, scores = signal_at(t)
        future = open_price[t + 1 + h] / open_price[t + 1] - 1
        values.append(rank_ic(scores, future))
    print(f"持有 {h:2d} 日: 平均截面 rank IC={np.nanmean(values): .4f}, 样本数={len(values)}")


从一天改成五天或二十天，就是信号期限诊断。它观察的是不同累计持有区间，不能完全等同于“把同一信号放旧几天再交易”的老化实验：持有期变长，收益会同时受到新信息与区间累积的影响。若要研究执行延迟，应固定持有期，例如五日，再把入场向后推 0、1、5、10 日，比较同一旧分数对后续五日收益的关系。两种曲线回答不同问题，报告时应把横轴写清楚。

Alpha 衰减（alpha decay）泛指预测能力随信息变旧而减弱的现象，不保证实测曲线单调下降。趋势可能需要一段时间才能体现，也可能样本噪声压过机制。图 5-5 特意保留不单调甚至为负的样本结果，提醒我们不要先画一条漂亮的指数曲线再称其为证据。

当 \(h>1\) 时，相邻决策的未来收益区间重叠。420 个日度相关系数不是 420 个相互独立的发现。应按时间块评估稳定性，使用与依赖结构相容的不确定性估计，或在不重叠抽样中观察结论；不能直接把普通独立样本标准误搬过来。多资产之间也共享共同风险，因此“资产数乘天数”通常夸大有效信息量。




![项目 A 合成面板上 rank IC 随前瞻持有期的衰减](figures/ch05_ic_horizon_decay.png)

**图 5-5｜不同持有期的实测 IC。** 用项目 A 的同一组动量/波动信号与不同持有期的未来收益，计算平均截面 rank IC。本次合成样本不呈单调衰减，部分持有期接近零或为负，因此不能直接套用正的指数衰减曲线。数据类型：固定 seed 项目输出，仅用于教学。


## 8. 从分数到权重：变换会改变我们交易的内容

截面标准分（z-score）把当日分数减去当日平均值，再除以当日标准差。它表达相对强弱而非绝对方向：四个资产都上涨时，低于均值者依然得到负分。排名保留次序但舍弃强弱差距；极值截断（winsorization）限制异常值对组合的影响，却也可能削弱真正强烈的信号。

图 5-6 将这些步骤分开。请回到我们的实现：`score` 用来比较和诊断；主线 `weights_at` 采用方向除以波动，只使用分数符号。也就是说，一个 0.1 和一个 3.0 的正分，方向信号相同。若改用连续分数决定仓位，就是新的组合映射，可能降低对弱趋势的持仓，也可能提高噪声造成的换手。不能把“信息系数没变”当成“收益曲线一定不变”。




![原始分数、截面 z-score 与截断归一化权重的三步转换](figures/ch05_score_to_weight.png)

**图 5-6｜从分数到仓位的三步。** 三图依次为原始分数、截面 z-score，以及截断分数再归一后的权重。本例结果位于图示 40% 参考线内；实际权重上限必须在最终归一后检查，不能只靠分数截断保证。数据类型：六个手写分数的概念示例。


中性化（neutralization）是去掉某些已知暴露。例如同一市场股票中，可以先用行业与规模解释原始信号，再用残差排序；在权重层也可以约束行业净暴露或市场贝塔。残差表示相对这些控制变量的额外信息，不是天然更好的预测。若利润增长的经济机制恰好通过行业趋势起作用，行业中性化会删除一部分真实故事，同时还可能增加交易。

标准化参数的时间口径也不同。当日截面均值来自当日已经可见的四个分数，可以合法用于当日决策；用完整历史样本的均值和标准差去标准化早期日期，则让未来进入过去。对主线来说，20 日波动窗口在决策日收盘结束；暖启动期至少需要 60 日价格，之前保持空仓，而不是往过去补出一条未来才知道的趋势。

## 9. 再量策略：收益、换手、基准和做空成本

主线决策 \(t\) 的权重 \(w_t\) 只乘下一开盘到再下一开盘的收益。假设成交完全、资金规模足够小，毛收益是资产收益的加权和。交易成本按目标权重绝对变化的和计算；借券费按空头名义敞口计提。借券年费率设为 2%，只是教学情景，真实借券费可能随资产和时间剧烈变化，还可能完全借不到。

这里暂用目标权重变化近似换手，没有精确处理价格漂移、保证金和自融资现金账本。这样可以保持最小实现透明，第六章再解决成交与现金。等权基准也从零仓位开始，在相同区间持有并承担调仓成本；它只有多头，因此没有借券费。主线若在样本末日仍有持仓，报告将其按最后可见开盘清仓并扣一次成本。


In [ ]:
def backtest(lookback=60, mode="trend", cost_bps=5.0, start=120, stop=None,
             borrow_annual=0.02):
    stop = n_days - 1 if stop is None else min(stop, n_days - 1)
    times = np.arange(max(start, lookback), stop)
    weights = np.array([weights_at(t, lookback, mode) for t in times])
    future = np.array([open_price[t + 2] / open_price[t + 1] - 1 for t in times])
    previous = np.vstack([np.zeros(n_assets), weights[:-1]])
    turnover = np.abs(weights - previous).sum(axis=1)
    gross = np.sum(weights * future, axis=1)
    trading_cost = turnover * cost_bps * 1e-4
    borrow = np.maximum(-weights, 0).sum(axis=1) * borrow_annual / 252
    net = gross - trading_cost - borrow
    # 在最后一个已记账区间结束后清仓；费用作用于此时剩余权益。
    terminal_cost = np.abs(weights[-1]).sum() * cost_bps * 1e-4
    net[-1] = (1 + net[-1]) * (1 - terminal_cost) - 1
    benchmark_gross = future.mean(axis=1)
    benchmark_net = benchmark_gross.copy()
    benchmark_net[0] -= cost_bps * 1e-4
    benchmark_net[-1] = (1 + benchmark_net[-1]) * (1 - cost_bps * 1e-4) - 1
    return {"times": times, "weights": weights, "future": future,
            "turnover": turnover, "gross": gross, "trade_cost": trading_cost,
            "borrow": borrow, "net": net, "benchmark": benchmark_net}

def describe(returns):
    wealth = np.cumprod(1 + returns)
    drawdown = wealth / np.maximum.accumulate(np.r_[1.0, wealth])[1:] - 1
    vol = returns.std(ddof=1)
    sharpe = returns.mean() / vol * np.sqrt(252) if vol > 0 else np.nan
    return {"累计收益": float(wealth[-1] - 1), "年化波动": float(vol * np.sqrt(252)),
            "Sharpe": float(sharpe), "最大回撤": float(drawdown.min())}

result = backtest()
for label, values in [("动量毛收益", result["gross"]), ("动量净收益", result["net"]),
                      ("等权净收益", result["benchmark"])]:
    print(label, {k: round(float(v), 4) for k, v in describe(values).items()})
print("每日平均单边换手:", round(float(result["turnover"].mean()), 4))


Sharpe 比率（夏普比率）在本例按日均收益除以日收益标准差，再乘 \(\sqrt{252}\) 计算，现金利率暂设为零。它衡量每单位历史波动对应的平均超额收益，不是胜率；独立同分布和方差稳定近似不成立时，平方根年化也只是描述性口径。最大回撤是净值相对此前峰值的最大跌幅，路径顺序很重要，不能由均值和方差完整推导。

胜率（hit rate）衡量方向判断正确的比例，但一次大亏可以抵消许多小赚。高低组收益差（spread return）衡量按分数选择赢家与输家是否产生收益；分组单调性检查得分由低到高时未来收益是否逐步改善。本例只有四资产，分组很粗，应该把每个资产的收益也列出来，而不是用一条平均曲线掩盖全部贡献。

## 10. 诊断差结果：先找发生在哪一层

第一层检查时间。若把 `future` 改成当天收盘前已发生的收益，结果回答的就不是原问题；如果去掉暖启动，可能读取不足窗口或负索引。第二层检查信号。逐期 IC 是否只被几个日期驱动？股票一类是否贡献了全部关系？第三层检查映射。单资产上限是不是在最终归一化后仍成立？信号方向是否被意外反转？第四层才检查成本与组合，避免因为成本后亏损就急着增加复杂模型。

下面把同一策略的毛贡献按资产展开，并在前后两个连续时间段观察净结果。两个区间不是事后挑选的好坏年份，而是固定的时间边界。分段净值使用整条策略的真实连续仓位，因此后一段不额外假装重新建仓；这种口径适合定位损益来源，但不等于独立启动的新回测。


In [ ]:
asset_contribution = np.mean(result["weights"] * result["future"], axis=0) * 252
for a, contribution in zip(assets, asset_contribution):
    print(a, f"年化算术毛贡献={contribution:.3%}")
for left, right in [(120, 270), (270, n_days - 1)]:
    selected = (result["times"] >= left) & (result["times"] < right)
    print(f"连续策略区间 [{left}, {right}):", describe(result["net"][selected]))

# 这里只做风险归因的最小示范：等权收益是一个因子，不能代表全部金融风险。
factor = result["benchmark"]
X = np.column_stack([np.ones(len(factor)), factor])
intercept, beta = np.linalg.lstsq(X, result["net"], rcond=None)[0]
print(f"相对等权因子: 日截距={intercept:.6f}, beta={beta:.3f}")


回看前面的主线输出，日频 IC 很弱、五至二十日的平均 IC 较高，但 60 日窗口的净收益依然明显落后等权。二者并不矛盾：IC 只衡量四资产的相对排序，主线还采用方向与逆波动配置，承担借券和每日换手成本；等权则在某些市场上涨时持续持有。诊断的作用是把差异解释到这些具体环节，而不是用一个指标否定另一个。

最后的回归形式是 \(r_t=\alpha+\beta f_t+\varepsilon_t\)。\(r_t\) 是动量净收益，\(f_t\) 是等权组合收益，\(\alpha\) 是这个特定单因子模型的日截距，\(\beta\) 是线性敏感度，\(\varepsilon_t\) 是未解释波动。这里没有报告显著性，也没有把截距直接称为可交易 Alpha：样本合成、因子不完整、参数可能随状态变化。换一组合理风险因子，截距可能改变，这正说明 Alpha 必须连同参照模型说明。

## 11. 成本、期限和研究选择必须一起看

回看长度、持有期限、再平衡频率是三个独立选择。主线回看 60 日，每日重新决策，持有一个开盘区间；它不是每 60 日才交易一次。信号虽慢，波动估计和仓位上限仍可能导致每日调整。改成每五日再平衡能降低换手，也可能让突发反转后的旧仓位多保留几天；它应作为明确的新规则比较。

下面固定共同起点 120，列出 20、60、120 日三个窗口与四档成本。共同起点避免短窗口因为多赚一段不同历史而获胜。我们保留整张表，不自动选择最高 Sharpe。若未来真的用这张表挑窗口，就应只在训练与验证期挑，随后在未参与选择的数据中冻结检验；同一段结果不能同时充当选择依据和独立证据。


In [ ]:
for lookback in [20, 60, 120]:
    for cost in [0, 5, 20, 50]:
        candidate = backtest(lookback=lookback, cost_bps=cost)
        values = describe(candidate["net"])
        print(f"窗口={lookback:3d}, 成本={cost:2d} bp, "
              f"累计={values['累计收益']:.2%}, Sharpe={values['Sharpe']:.3f}")


固定价格和权重路径时，提高非负交易费率会逐期降低净收益，因而不会提高净值。但 Sharpe 是均值与波动的比值，没有相同的普遍单调性。若成本参与最优仓位、资金约束或实际成交，权重路径也会改变，更不能把两条结果直接当作只改费用的实验。主线的参数表只改变费率，方向与权重都固定，属于最容易解释的敏感性分析。

容量（capacity）则进一步问资金变大后会怎样。同样的 20% 权重，对十万元与十亿元账户代表完全不同的订单。冲击、可借券数量和订单占市场成交量比例会改变成本甚至改变能否成交，所以“5 bp 下可盈利”并不回答资金上限。下一章会把抽象权重变成股数，再看未成交与机会成本。

## 12. 从研究机制到可交付结论

一个完整结论可以沿主线写成：我们假设渐进调仓造成中期方向延续；用四类资产的 60 日累计收益定方向、20 日历史波动定规模；信号收盘形成，下一个开盘成交；报告相对同资产等权的净收益、资产贡献、条件暴露、换手和成本敏感性；在急速反转、共同风险上升和流动性恶化时预期失效；真正部署前还缺少历史可交易产品、融资与逐单成交证据。

增加机器学习（machine learning）时，也应围绕同一问题判断增量。可以用非线性模型预测未来收益，但输入仍需时点可见，标签仍是相同开盘区间，模型必须在过去拟合，并相对这个简单方向基线报告成本后改善。模型准确率提高而换手翻倍，可能并不值得；增加许多参数却只改善训练期，也可能只是搜索噪声。

把多个信号组合前，要比较它们的净收益相关性、共同尾部和边际贡献，而不只看原始分数相关性。两个分数高度相关，经过不同风险约束可能产生不同收益；两个分数日常相关很低，也可能在危机里一起亏损。相关性矩阵只是起点，压力状态与执行需求是否同时集中，才决定组合是否有韧性。

## 13. 常见误解

**“趋势就是截面动量。”** 主线全市场下跌时可以全部做空；截面规则仍会选出相对赢家。它们回答不同问题，净暴露也不同。

**“排名标准化只是技术处理。”** 当四个资产都上涨时，排名减均值会主动做空较弱资产，已经改变经济押注。每次变换都要追问它增加了什么假设。

**“IC 高就应该提高杠杆。”** IC 不含收益幅度、相关风险、可成交性和成本。四资产样本的高 IC 尤其可能不稳定。

**“加入行业或市场中性化一定提高质量。”** 中性化是选择要保留与删除的风险来源，可能删掉有效机制并增加换手。应比较约束前后的成本与研究目标。

**“动量在合成样本有效就验证了机制。”** 合成数据的趋势来自我们自己写入的生成规则。它检验实现能否捕捉设定结构，真实机制必须用独立市场数据研究。

## 14. 手工练习

### 练习一：同一组涨跌，两种不同问题

四资产过去收益为 12%、8%、4%、1%，下一期收益为 3%、2%、-1%、1%。先忽略波动差异与成本。时间序列动量各持 +25%；截面策略只买第一名 +50%、卖最后一名 -50%。分别算毛收益、总敞口与净敞口。

<details><summary>参考答案</summary>

时间序列规则毛收益为 \((3+2-1+1)/4=1.25\%\)，总敞口与净敞口都是 1。截面规则毛收益为 \(0.5\times3\%-0.5\times1\%=1\%\)，总敞口 1、净敞口零。两者都可能盈利，但前者承担整体上涨暴露，后者主要比较赢家与输家的相对变化。

</details>

### 练习二：信号没有变，仓位为什么动了

两资产动量始终为正。昨天日波动分别是 1% 与 2%，今天变成 2% 与 2%。忽略单名上限，按方向除以波动、再按总敞口 1 归一。计算两天权重、目标权重近似换手以及 10 bp 下成本。

<details><summary>参考答案</summary>

昨天临时量 100 与 50，权重为 2/3 和 1/3；今天为 50 与 50，权重各 1/2。换手为 \(|1/2-2/3|+|1/2-1/3|=1/3\)，成本为 \((1/3)\times0.1\%\approx0.0333\%\)。方向不变不等于无需交易；风险缩放也会带来换手。

</details>

### 练习三：正 IC 为什么仍可亏损

某多空组合每次按实际权重折算到整个账户的毛收益平均为 8 bp（不是未经权重缩放的高组减低组收益差）。一次完整调仓需要卖出旧多头、买入新多头、买回旧空头、卖出新空头，总成交名义金额为账户权益的 2 倍。单边成本 5 bp。忽略融资和借券，账户净收益大约是多少？

<details><summary>参考答案</summary>

交易成本为 \(2\times5=10\) bp，净收益约 -2 bp。排序确实可能有预测力，但毛幅度不够覆盖实现这次排序所需的交易。还可以研究降低调仓频率、提高交易门槛或更平滑的仓位，而不是直接认为必须换一个更复杂模型。

</details>

## 15. 自测与思考

1. 为什么主线既保留 `score`，又用 `sign(momentum)` 产生实际方向？
2. 将四资产的所有未来收益同时加 1%，当期 rank IC 与多头策略收益如何变化？
3. 为什么不同持有期的 IC 曲线不能自动决定最优再平衡频率？
4. 主线相对等权回归截距为正，离“存在可部署 Alpha”还差哪些证据？

<details><summary>参考答案</summary>

1. 连续分数用于比较相对强弱和预测关系，方向基线刻意避免引入额外幅度映射。若用分数大小分配仓位，应单独研究变化后的收益与换手。
2. 排名不变，因此 rank IC 不变；多头策略会因净敞口获得额外收益，净敞口为零的多空组合则不会。这说明截面指标不描述全部方向收益。
3. IC 不含实际换手、持仓重叠、收益幅度、冲击和延迟。不同期限还使用重叠标签，统计不确定性不同。
4. 需要真实且时点一致的可交易数据、合理的风险因子与状态归因、未参与选择的样本外表现、完整成本与融资、容量和极端状态证据。一个特定回归模型中的正截距只回答很窄的问题。

</details>

## 16. 本章小结

从一条 60 日多资产方向规则出发，我们依次问了为什么可能延续、如何比较不同资产、分数怎样变仓位、预测质量怎样测、收益由谁贡献、成本与规模怎样改变结果。同一条主线让每个概念都有落点：IC 对应分数与未来收益，换手对应目标变化，Beta 对应风险暴露，Alpha 必须连同基准与约束解释。

一项好的策略研究不是记住更多策略名字，而是能对自己的规则完整说明机制、时间、比较对象和失败边界。下一章会继续使用目标仓位的语言，但将每个权重拆成订单、实际成交和现金，让“想持有”与“最终持有”的差异变得可计算。
